In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, IntegerType, ArrayType, DateType
import sys
import os
from delta import DeltaTable
from pyspark.sql import DataFrame
from pyspark.sql.utils import AnalysisException
from delta.tables import *
import io
import json

In [0]:
def create_spark_session():
    return SparkSession \
        .builder \
        .appName("File Streaming Demo") \
        .master("local[3]") \
        .config("spark.databricks.delta.schema.autoMerge.enabled", "true")\
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .enableHiveSupport()\
        .getOrCreate()

In [0]:
def create_deltaTable_insert_update_rows(spark:SparkSession,columns:list, location:str,merge_condition:str,df:DataFrame):
    if (DeltaTable.isDeltaTable(spark, location)):
        print('tabela delta existente')
        deltaTable = DeltaTable.forPath(spark, location)
        deltaTable.alias('tgt') \
            .merge(
                df.alias('src'),
                merge_condition
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
    else:
        print('tabela delta inexistente')    
        DeltaTable \
            .create(spark) \
            .addColumns(columns) \
            .location(location) \
            .execute()
        deltaTable = DeltaTable.forPath(spark, location)
        deltaTable.alias('tgt') \
            .merge(
                df.alias('src'),
                merge_condition
            ) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()


#### Caminhos para a camada silver

In [0]:
path_silver_dengue= '/FileStore/silver/dados_degue/casos_dengue'
path_silver_chuva= '/FileStore/silver/dados_degue/chuvas'


#### Leitura da dos dados em dataframes

In [0]:
df_dengue = spark.read.format('delta').load(path_silver_dengue)

In [0]:
df_chuva = spark.read.format('delta').load(path_silver_chuva)

#### Criação do banco de dados dengue_chuvas

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS dengue_chuvas")

Out[7]: DataFrame[]

#### Analise soma casos dengue agrupado por 'ano','mes','estado','cidade'

In [0]:
df_media_dengue = df_dengue.groupBy('ano','mes','estado','cidade').agg(
    f.sum('quantidade_casos').alias('soma_casos_dengue')
    )

df_media_dengue.write.format('delta').mode('overwrite').saveAsTable('dengue_chuvas.media_dengue')

consulta_dengue = "SELECT * FROM dengue_chuvas.df_media_dengue"
df_resultado_chuvas = spark.sql(consulta_dengue)

df_resultado_chuvas.display()

ano,mes,estado,cidade,soma_casos_dengue
2015,12,ES,Cachoeiro De Itapemirim,1027.0
2019,7,ES,Cachoeiro De Itapemirim,1196.0
2016,4,ES,Conceição Da Barra,16.0
2015,5,ES,Conceição Do Castelo,2.0
2017,4,ES,Conceição Do Castelo,2.0
2015,5,ES,Divino De São Lourenço,0.0
2018,5,ES,Iúna,0.0
2015,4,ES,Jaguaré,18.0
2019,2,ES,Laranja Da Terra,27.0
2017,6,ES,Linhares,48.0


#### Analise soma quantidade de chuva agrupado por 'ano','mes','estado'

In [0]:
df_media_chuva = df_chuva.groupBy('ano','mes','estado').agg(
    f.sum('mm').alias('soma_chuva')
)
df_media_chuva.write.format('delta').mode('overwrite').saveAsTable('dengue_chuvas.media_chuva')

consulta_chuvas = "SELECT * FROM dengue_chuvas.df_media_chuva"
df_resultado_chuvas = spark.sql(consulta_chuvas)

df_resultado_chuvas.display()

ano,mes,estado,soma_chuva
2018,6,GO,3.2000000000000006
2018,2,BA,4142.599999999954
2018,3,SP,6245.199999999868
2018,11,BA,4865.999999999928
2018,11,RJ,5465.399999999914
2018,6,AM,2314.0000000000064
2018,2,RN,922.6000000000013
2018,4,PR,592.399999999999
2018,2,SP,5156.39999999991
2018,9,TO,417.79999999999956


#### Analise analise_chuvas_dengue 'ano','mes','estado'

In [0]:
df_media_dengue =  df_dengue.groupBy('ano','mes','estado').agg(
    f.sum('quantidade_casos').alias('soma_casos_dengue')
)  #.filter((f.col('estado') == 'SP') & (f.col('ano') == '2015') & (f.col('mes') == '1'))

In [0]:
df_media_chuva = df_chuva.groupBy('ano','mes','estado').agg(
    f.abs(f.sum('mm')).alias('soma_chuva')
)#.filter((f.col('estado') == 'SP') & (f.col('ano') == '2015') & (f.col('mes') == '1'))

In [0]:
df_condicao = (df_media_dengue.ano == df_media_chuva.ano) & (df_media_dengue.mes == df_media_chuva.mes) & (df_media_dengue.estado == df_media_chuva.estado)

df_analise = df_media_dengue.join(df_media_chuva, condicao, 'inner') \
    .drop(df_media_dengue.ano, df_media_dengue.mes, df_media_dengue.estado) \
    .withColumn('casos_Dengue/ChuvaAcum', f.col('soma_casos_dengue') / f.col('soma_chuva'))

df_analise.display()


soma_casos_dengue,ano,mes,estado,soma_chuva,casos_Dengue/ChuvaAcum
481.0,2018,3,SP,6245.199999999868,0.07701915070774518
507.0,2018,11,RJ,5465.399999999914,0.09276539686024957
2446.0,2018,4,PR,592.399999999999,4.128966914247138
372.0,2018,2,SP,5156.39999999991,0.07214335582964986
1032.0,2018,12,SP,5207.1999999999425,0.19818712551851503
0.0,2018,11,RS,7293.399999999929,0.0
1221.0,2018,8,ES,1028.000000000003,1.1877431906614753
30.0,2018,7,MG,293.59999999999894,0.10217983651226195
2084.0,2018,2,PR,2724.2000000000003,0.7649952279568313
635.0,2018,9,CE,23.999999999999993,26.458333333333343


In [0]:
df_analise.write.format('delta').mode('overwrite').saveAsTable('dengue_chuvas.analise')

consulta_analise = "SELECT * FROM dengue_chuvas.analise"
df_resultado_analise = spark.sql(consulta_analise)

df_resultado_analise.display()

soma_casos_dengue,ano,mes,estado,soma_chuva,casos_Dengue/ChuvaAcum
481.0,2018,3,SP,6245.199999999868,0.07701915070774518
507.0,2018,11,RJ,5465.399999999914,0.09276539686024957
2446.0,2018,4,PR,592.399999999999,4.128966914247138
372.0,2018,2,SP,5156.39999999991,0.07214335582964986
1032.0,2018,12,SP,5207.1999999999425,0.19818712551851503
0.0,2018,11,RS,7293.399999999929,0.0
1221.0,2018,8,ES,1028.000000000003,1.1877431906614753
30.0,2018,7,MG,293.59999999999894,0.10217983651226195
2084.0,2018,2,PR,2724.2000000000003,0.7649952279568313
635.0,2018,9,CE,23.999999999999993,26.458333333333343
